# DistCp Partition-Filter Scoped Validation — Behavior Test (WF-201)

Validates the post-DistCp file/size match logic in `migration_dag_mapr_to_s3.py` after the
WF-201 fix. The fix changes how S3 destination metrics are measured when a `partition_filter`
is active.

### What changed

**Before WF-201:** when a partition filter was active, the source baseline
(`source_file_count`, `source_total_size_bytes`) was scoped to the filtered partitions on
MapR, but the S3 measurements (`s3_*_after`, `s3_*_transferred`) were computed at the **table
root** prefix on S3. The CASE-based comparison in `update_distcp_status` then compared a
per-partition source baseline against a table-level S3 metric — false negatives on incremental runs.

**After WF-201:** the bash that runs over SSH defines a `sum_s3_metrics_over_paths` helper
and a `DEST_PART_PATHS` array. `S3_BEFORE` and `S3_AFTER` are the **sum of per-partition
snapshots**, scoped to exactly the partitions that were filtered. The CASE collapses to a
single direct equality on the snapshot.

### What this notebook tests

| Phase | Scenario | Expectation |
|---|---|---|
| Step 3 | Run the scoped helper against real S3 | Sum equals per-partition `getContentSummary` baseline |
| Step 4 | Cross-check assertions | Asserts scoped equals per-partition baseline |
| Step 5 | Incremental run | New partition copied; filter scopes the run to it. Scoped measurement covers only that partition; table-root covers all partitions including prior ones |

### Prerequisites

- JupyterHub Spark session with Iceberg and S3 access (NX1 default).
- `hadoop fs` CLI configured for the same S3 bucket (used by the DAG over SSH).

This notebook exercises only the S3 side. The MapR side and DistCp itself are mocked via SSH stubs;
the destination is simulated by writing Parquet directly to S3 (which is what DistCp would have
produced). End-to-end validation against MapR still requires a staging-cluster smoke run.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────

BUCKET         = "nx1poc-pdc-default-ygglold"
TENANT_PREFIX  = f"s3a://{BUCKET}/es-tenant-2"
TEST_BASE      = f"{TENANT_PREFIX}/test_distcp_partition_filter_scope"

TEST_DB        = "test_wf201"
TEST_TABLE     = "events"
S3_LOC         = f"{TEST_BASE}/{TEST_DB}/{TEST_TABLE}"

# Filter scope: the three partitions that the simulated DistCp run would copy.
FILTERED_PARTITIONS = ["dt=2026-04-01", "dt=2026-04-02", "dt=2026-04-03"]

# Used in Step 5 (incremental-run scenario): the partition that arrives in a later
# DAG run after the initial migration, with partition_filter scoping the run to it.
INCREMENTAL_PARTITION = "dt=2026-04-04"

print(f"S3 location:         {S3_LOC}")
print(f"Filtered partitions: {FILTERED_PARTITIONS}")
print(f"Incremental partition: {INCREMENTAL_PARTITION}")

In [ ]:
from pyspark.sql import SparkSession

try:
    _ = spark
    print(f"Using existing Spark session (version {spark.version})")
except NameError:
    spark = SparkSession.builder \
        .appName("test-distcp-partition-filter-scope") \
        .enableHiveSupport() \
        .getOrCreate()
    print(f"Created Spark session (version {spark.version})")

spark.sparkContext.setLogLevel("WARN")

# ── S3 helpers  ───────────────────────────────────
from py4j.java_gateway import java_import
java_import(spark._jvm, "org.apache.hadoop.fs.*")

def _fs(path):
    return spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jvm.java.net.URI(path), spark._jsc.hadoopConfiguration()
    )

def s3_delete(path):
    fs = _fs(path); p = spark._jvm.org.apache.hadoop.fs.Path(path)
    if fs.exists(p):
        fs.delete(p, True); print(f"  Deleted: {path}")
    else:
        print(f"  Not found (skip): {path}")

def s3_content_summary(path):
    """Mirror what discovery uses on the source side: getContentSummary on the path."""
    fs = _fs(path); p = spark._jvm.org.apache.hadoop.fs.Path(path)
    if not fs.exists(p):
        return {"file_count": 0, "total_size": 0}
    cs = fs.getContentSummary(p)
    return {"file_count": int(cs.getFileCount()), "total_size": int(cs.getLength())}

---
## Step 0: Bootstrap Hadoop CLI

The bash helpers in `migration_dag_mapr_to_s3.py` invoke `hadoop fs` over SSH on a data-edge
node. JupyterHub here doesn't ship the Hadoop CLI, so we install it locally and point it at
the same s3a config Spark is using.

This cell is idempotent: download + extract happen once and the cell becomes a no-op on
subsequent runs. Subsequent steps' `subprocess.run` calls inherit `PATH` / `HADOOP_HOME` from
`os.environ`, so the CLI is available everywhere downstream without repeating any setup.

In [ ]:
# ── Bootstrap Hadoop CLI on this JH node ─────────────────────────────────────
# Downloads Hadoop 3.3.6 (~700 MB, one-time) to $HOME/hadoop-3.3.6/, generates
# core-site.xml from the running Spark session's hadoopConfiguration(), and
# puts NX1's custom credentials-provider JAR on HADOOP_CLASSPATH so `hadoop fs`
# can resolve it. Subsequent subprocess.run calls inherit PATH / HADOOP_HOME /
# HADOOP_CLASSPATH from os.environ. Re-running this cell is a no-op once
# download + extract are done.

import os, subprocess, urllib.request, tarfile, pathlib, glob
from xml.sax.saxutils import escape as xml_escape

HADOOP_VER         = "3.3.6"
SPARK_HOME_DEFAULT = "/home/amaluga/utils/spark"  # fallback if $SPARK_HOME is unset

HOME        = pathlib.Path.home()
HADOOP_HOME = HOME / f"hadoop-{HADOOP_VER}"
TARBALL     = HOME / f"hadoop-{HADOOP_VER}.tar.gz"
TARBALL_URL = f"https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VER}/hadoop-{HADOOP_VER}.tar.gz"
SPARK_JARS  = pathlib.Path(os.environ.get("SPARK_HOME", SPARK_HOME_DEFAULT)) / "jars"

if not HADOOP_HOME.exists():
    if not TARBALL.exists():
        print(f"Downloading {TARBALL_URL} (~700 MB, one-time)...")
        try:
            urllib.request.urlretrieve(TARBALL_URL, TARBALL)
        except Exception:
            if TARBALL.exists(): TARBALL.unlink()
            raise
        print(f"  -> {TARBALL}")
    print(f"Extracting to {HADOOP_HOME}...")
    with tarfile.open(TARBALL) as tf:
        tf.extractall(HOME)
else:
    print(f"Using existing {HADOOP_HOME}")

java_home = os.environ.get("JAVA_HOME") or subprocess.run(
    ["bash", "-c", "readlink -f $(command -v java) | xargs dirname | xargs dirname"],
    capture_output=True, text=True
).stdout.strip()
print(f"JAVA_HOME = {java_home}")

# Pull s3a-relevant config from the live Spark session into core-site.xml.
# Includes NX1's custom JCEKS-dir key so the bare CLI looks at the same path
# Spark uses (default in the provider class is /opt/spark/conf/jceks, which
# may not exist on a JH node).
hconf = spark._jsc.hadoopConfiguration()
S3A_KEYS = [
    "fs.s3a.access.key", "fs.s3a.secret.key", "fs.s3a.session.token",
    "fs.s3a.endpoint", "fs.s3a.endpoint.region", "fs.s3a.path.style.access",
    "fs.s3a.connection.ssl.enabled", "fs.s3a.aws.credentials.provider",
    "fs.s3a.signing-algorithm",
    "fs.s3a.impl", "fs.AbstractFileSystem.s3a.impl",
    "hadoop.security.credential.provider.path",
    "fs.s3a.security.credential.provider.path",
    "fs.s3a.bucket.jceks.dir",
    f"fs.s3a.bucket.{BUCKET}.jceks.dir",
    f"fs.s3a.bucket.{BUCKET}.aws.credentials.provider",
    f"fs.s3a.bucket.{BUCKET}.security.credential.provider.path",
    f"fs.s3a.bucket.{BUCKET}.access.key",
    f"fs.s3a.bucket.{BUCKET}.secret.key",
    f"fs.s3a.bucket.{BUCKET}.endpoint",
]
exported = {}
for k in S3A_KEYS:
    v = hconf.get(k)
    if v not in (None, ""):
        exported[k] = v
exported.setdefault("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

print(f"Spark resolves fs.s3a.bucket.jceks.dir = "
      f"{hconf.get('fs.s3a.bucket.jceks.dir')!r}")
print(f"Spark resolves fs.s3a.bucket.{BUCKET}.jceks.dir = "
      f"{hconf.get(f'fs.s3a.bucket.{BUCKET}.jceks.dir')!r}")

HADOOP_CONF_DIR = HADOOP_HOME / "etc" / "hadoop"
HADOOP_CONF_DIR.mkdir(parents=True, exist_ok=True)
props_xml = "\n".join(
    f"  <property><name>{k}</name><value>{xml_escape(v)}</value></property>"
    for k, v in exported.items()
)
(HADOOP_CONF_DIR / "core-site.xml").write_text(
    f'<?xml version="1.0"?>\n<configuration>\n{props_xml}\n</configuration>\n'
)
print(f"Wrote {HADOOP_CONF_DIR/'core-site.xml'} with: {sorted(exported)}")

nx1_jars = sorted(glob.glob(str(SPARK_JARS / "nx1cloud-*.jar")))
print(f"NX1 jars on HADOOP_CLASSPATH: {nx1_jars}")

os.environ["HADOOP_HOME"]           = str(HADOOP_HOME)
os.environ["HADOOP_CONF_DIR"]       = str(HADOOP_CONF_DIR)
os.environ["JAVA_HOME"]             = java_home
os.environ["HADOOP_OPTIONAL_TOOLS"] = "hadoop-aws"
os.environ["HADOOP_CLASSPATH"]      = ":".join(nx1_jars)
os.environ["PATH"]                  = f"{HADOOP_HOME}/bin:" + os.environ.get("PATH", "")
# HADOOP_JCEKS_PASSWORD propagates automatically if it's already in os.environ.

ver = subprocess.run(["hadoop", "version"], capture_output=True, text=True)
print(ver.stdout.splitlines()[0] if ver.stdout else ver.stderr)

probe = subprocess.run(["hadoop", "fs", "-ls", f"s3a://{BUCKET}/"], capture_output=True, text=True)
print(f"\n`hadoop fs -ls s3a://{BUCKET}/` exit={probe.returncode}")
if probe.stdout: print("STDOUT:\n" + probe.stdout[:2000])
if probe.stderr: print("STDERR:\n" + probe.stderr[:2000])


---
## Step 1: Pre-cleanup

Removes any leftover state from prior runs — Iceberg HMS entries and the simulated S3 dataset.

In [ ]:
print("Pre-cleanup: removing state from prior runs...\n")

try:
    spark.sql(f"DROP TABLE IF EXISTS {TEST_DB}.{TEST_TABLE} PURGE")
    print(f"  Dropped table (if existed): {TEST_DB}.{TEST_TABLE}")
except Exception:
    pass

try:
    spark.sql(f"DROP DATABASE IF EXISTS {TEST_DB}")
    print(f"  Dropped database (if existed): {TEST_DB}")
except Exception:
    pass

s3_delete(TEST_BASE)
print("\nPre-cleanup done.")

---
## Step 2: Seed simulated post-DistCp data on S3

DistCp would have copied Parquet partition directories from MapR to `S3_LOC`. We simulate the
destination state by writing Parquet directly. Three partitions are written; their file counts
and byte sizes form the per-partition baseline used in Steps 3-5.

In [ ]:
from datetime import datetime

rows_per_part = {
    "dt=2026-04-01": [(1, "a"), (2, "b"), (3, "c")],
    "dt=2026-04-02": [(4, "d"), (5, "e")],
    "dt=2026-04-03": [(6, "f"), (7, "g"), (8, "h"), (9, "i")],
}

for part_str, rows in rows_per_part.items():
    df = spark.createDataFrame(rows, ["id", "label"])
    target = f"{S3_LOC}/{part_str}"
    df.coalesce(1).write.mode("overwrite").parquet(target)
    print(f"  wrote {target}")

print("\nSeeded 3 partitions on S3.")

In [ ]:
# ── Baseline measurements via getContentSummary (mirrors discovery's source-side path) ──────────────────────

per_part_baseline = {}
for p in FILTERED_PARTITIONS:
    per_part_baseline[p] = s3_content_summary(f"{S3_LOC}/{p}")

filtered_files = sum(b["file_count"] for b in per_part_baseline.values())
filtered_bytes = sum(b["total_size"] for b in per_part_baseline.values())

table_baseline = s3_content_summary(S3_LOC)

print("Per-partition baseline (= what discovery would store as source_file_count / source_total_size_bytes):")
for p, b in per_part_baseline.items():
    print(f"  {p}: files={b['file_count']:>3d}  bytes={b['total_size']:>10d}")
print(f"  ----- sum -----")
print(f"  TOTAL: files={filtered_files}  bytes={filtered_bytes}")
print()
print(f"Table-level baseline (= what calculate_s3_metrics_hadoop would return at table root):")
print(f"  files={table_baseline['file_count']}  bytes={table_baseline['total_size']}")

---
## Step 3: Run the scoped helper against real S3

Builds a measurement-only bash script that mirrors what the DAG ships over SSH (same
`calculate_s3_metrics_hadoop` and `sum_s3_metrics_over_paths` definitions), then executes it
locally against the seeded S3 dataset using the JH `hadoop fs` CLI. The result should match
the per-partition baseline computed in Step 2.

In [ ]:
import subprocess, shlex

MEASUREMENT_HELPER = r'''
calculate_s3_metrics_hadoop() {
    local location=$1
    if ! hadoop fs -test -d "$location" 2>/dev/null; then
        echo "S3_FILE_COUNT=0"; echo "S3_TOTAL_SIZE=0"; return
    fi
    FILE_COUNT=$(hadoop fs -ls -R "$location" 2>/dev/null | grep '^-' | wc -l)
    TOTAL_SIZE=$(hadoop fs -du -s "$location" 2>/dev/null | awk '{print $1}')
    [ -z "$FILE_COUNT" ] && FILE_COUNT=0
    [ -z "$TOTAL_SIZE" ] && TOTAL_SIZE=0
    echo "S3_FILE_COUNT=$FILE_COUNT"; echo "S3_TOTAL_SIZE=$TOTAL_SIZE"
}

sum_s3_metrics_over_paths() {
    local fc=0 ts=0 pfc pts result
    for path in "$@"; do
        result=$(calculate_s3_metrics_hadoop "$path")
        pfc=$(echo "$result" | grep "^S3_FILE_COUNT=" | cut -d'=' -f2)
        pts=$(echo "$result" | grep "^S3_TOTAL_SIZE=" | cut -d'=' -f2)
        [ -z "$pfc" ] && pfc=0
        [ -z "$pts" ] && pts=0
        fc=$((fc + pfc))
        ts=$((ts + pts))
    done
    echo "S3_FILE_COUNT=$fc"; echo "S3_TOTAL_SIZE=$ts"
}
'''

def measure_scoped(paths):
    """Run sum_s3_metrics_over_paths against the given S3 paths and parse output."""
    paths_quoted = ' '.join(shlex.quote(p) for p in paths)
    script = MEASUREMENT_HELPER + f'\nDEST_PART_PATHS=({paths_quoted})\n' \
             f'sum_s3_metrics_over_paths "${{DEST_PART_PATHS[@]}}"\n'
    out = subprocess.run(["bash", "-c", script], capture_output=True, text=True, check=True)
    parsed = dict(line.split("=", 1) for line in out.stdout.strip().splitlines() if "=" in line)
    return {"file_count": int(parsed["S3_FILE_COUNT"]), "total_size": int(parsed["S3_TOTAL_SIZE"])}

def measure_table_root(loc):
    """Pre-WF-201 behavior: single call against the table prefix."""
    script = MEASUREMENT_HELPER + f'\ncalculate_s3_metrics_hadoop {shlex.quote(loc)}\n'
    out = subprocess.run(["bash", "-c", script], capture_output=True, text=True, check=True)
    parsed = dict(line.split("=", 1) for line in out.stdout.strip().splitlines() if "=" in line)
    return {"file_count": int(parsed["S3_FILE_COUNT"]), "total_size": int(parsed["S3_TOTAL_SIZE"])}

scoped = measure_scoped([f"{S3_LOC}/{p}" for p in FILTERED_PARTITIONS])
print(f"Scoped measurement (new logic): {scoped}")
print(f"Per-partition baseline:         {{'file_count': {filtered_files}, 'total_size': {filtered_bytes}}}")

---
## Step 4: Cross-check against the per-partition baseline

The new scoped measurement must equal the per-partition `getContentSummary` sum. If they
diverge, the bash helper has a bug (or the seed in Step 2 differs from what's currently on S3).

In [ ]:
assert scoped["file_count"] == filtered_files, \
    f"file_count mismatch: scoped={scoped['file_count']} vs baseline={filtered_files}"
assert scoped["total_size"] == filtered_bytes, \
    f"total_size mismatch: scoped={scoped['total_size']} vs baseline={filtered_bytes}"

print("PASS — scoped measurement equals per-partition baseline.")
print(f"  files: {scoped['file_count']}")
print(f"  bytes: {scoped['total_size']}")

---
## Step 5: Incremental run scenario

The first migration copied `dt=2026-04-01` through `dt=2026-04-03` (the data seeded in
Step 2 represents that prior state on S3). A subsequent DAG run with `partition_filter`
scoped to `dt=2026-04-04` only must compare a per-partition source baseline against a
per-partition S3 measurement.

Discovery measures the source baseline for `dt=2026-04-04` only — `source_file_count`
and `source_total_size_bytes` reflect just that partition. The S3-side measurement has
to be scoped the same way.

- **Scoped (new) measurement**: covers `s3a://.../dt=2026-04-04/` only.
- **Table-root (old) measurement**: `hadoop fs -du -s` on the table prefix. Returns the
  cumulative state of all four partitions — the three from prior runs plus this one —
  and disagrees with the single-partition source baseline.


In [ ]:
# ── Simulate the incremental DAG run for dt=2026-04-04 ─────────────────────────────
# DistCp would have copied this single partition from MapR to S3. We seed it directly.

incremental_rows = [(100, "x"), (101, "y"), (102, "z"), (103, "w"), (104, "v")]
df_incremental = spark.createDataFrame(incremental_rows, ["id", "label"])
incremental_path = f"{S3_LOC}/{INCREMENTAL_PARTITION}"
df_incremental.coalesce(1).write.mode("overwrite").parquet(incremental_path)
print(f"  wrote {incremental_path}")

# Discovery on the MapR side would call getContentSummary on this partition path only.
# That's the source baseline the comparison will use.
incremental_baseline = s3_content_summary(incremental_path)
print(f"  source baseline (this run): files={incremental_baseline['file_count']}  bytes={incremental_baseline['total_size']}")

INCREMENTAL_FILTER = [INCREMENTAL_PARTITION]


In [ ]:
# Run both the scoped (new) and table-root (old) measurements. The filter for this
# run is just the one incremental partition.

scoped_inc     = measure_scoped([f"{S3_LOC}/{p}" for p in INCREMENTAL_FILTER])
table_root_inc = measure_table_root(S3_LOC)

print("Source baseline (discovery, scoped to this run's filter):")
print(f"  files={incremental_baseline['file_count']}  bytes={incremental_baseline['total_size']}")
print()
print(f"S3 measurement (NEW, scoped):     files={scoped_inc['file_count']:>3d}  bytes={scoped_inc['total_size']}")
print(f"S3 measurement (OLD, table-root): files={table_root_inc['file_count']:>3d}  bytes={table_root_inc['total_size']}")
print()

# Scoped equals the single-partition source baseline.
assert scoped_inc == incremental_baseline, (
    f"Scoped measurement disagrees with single-partition source baseline: "
    f"{scoped_inc} vs {incremental_baseline}"
)
print("PASS — scoped S3 measurement equals the single-partition source baseline.")

# Table-root sums in the prior partitions and won't equal the single-partition baseline.
assert table_root_inc["file_count"]  > incremental_baseline["file_count"]
assert table_root_inc["total_size"]  > incremental_baseline["total_size"]
extra_files = table_root_inc['file_count'] - incremental_baseline['file_count']
extra_bytes = table_root_inc['total_size'] - incremental_baseline['total_size']
print(f"      table-root includes prior partitions: +{extra_files} files / +{extra_bytes} bytes")
print()
print("update_distcp_status comparison for this incremental run:")
print(f"  source_file_count            = {incremental_baseline['file_count']:>3d}    (per-partition discovery on MapR)")
print(f"  s3_file_count_after  (NEW)   = {scoped_inc['file_count']:>3d}    -> file_count_match = TRUE")
print(f"  s3_file_count_after  (OLD)   = {table_root_inc['file_count']:>3d}    -> file_count_match = FALSE  (pre-WF-201)")


---
## Step 6: Cleanup

Removes all S3 artifacts seeded by this notebook. Safe to re-run from the top after this cell.

In [ ]:
print("Cleaning up all test artifacts...\n")

try:
    spark.sql(f"DROP TABLE IF EXISTS {TEST_DB}.{TEST_TABLE} PURGE")
except Exception:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {TEST_DB}.{TEST_TABLE}")
    except Exception as e:
        print(f"  Skip {TEST_DB}.{TEST_TABLE}: {e}")

try:
    spark.sql(f"DROP DATABASE IF EXISTS {TEST_DB}")
    print(f"  Dropped database: {TEST_DB}")
except Exception as e:
    print(f"  Skip database: {e}")

s3_delete(TEST_BASE)
print("\nCleanup complete.")